In [ ]:
# Test harness for clean_pipeline

import numpy as np
import pandas as pd

# Function to format config summary ouput into something readable!

def _norm(x):
    """Recursively normalise numpy scalars and nested dict/list structures."""
    
    # numpy scalar → Python scalar
    if isinstance(x, (np.integer, np.floating, np.bool_)):
        return x.item()

    # pandas NA → None
    if x is pd.NA:
        return None

    # dict → normalise each value
    if isinstance(x, dict):
        return {k: _norm(v) for k, v in x.items()}

    # list/tuple → normalise each element
    if isinstance(x, (list, tuple)):
        return [_norm(v) for v in x]

    # everything else unchanged
    return x


def format_pipeline_summary(summary):

    # Convert numpy scalars to native Python types for readability
    def _norm_scalar(x):

        if isinstance(x, (np.integer, np.floating, np.bool_)):
            return x.item()
        if x is pd.NA:
            return None
        return x

    # Format the clean_pipeline summary dictionary into a readable text block, normalizing numpy scalars
    lines = []
    add = lines.append
    add("=== CLEANING PIPELINE SUMMARY ===\n")

    # Steps run / skipped
    steps_run = list(summary.get('steps run', []))
    steps_skipped = list(summary.get('steps skipped', []))
    add("Steps executed:")
    for step in steps_run:
        add(f"  • {step}")
    add("\nSteps skipped:")
    for step in steps_skipped:
        add(f"  • {step}")

    # Parameters passed
    add("\n--- PARAMETERS PASSED ---")
    params = summary.get('parameters passed', {})
    for step, p in params.items():
        add(f"\n[{step}]")
        for k, v in p.items():
            add(f"  {k}: {v}")

    # Cleaning function outputs
    add("\n--- CLEANING FUNCTION OUTPUTS ---")
    outputs = summary.get('cleaning function outputs', {})
    for step, out in outputs.items():
        add(f"\n[{step}]")
        if not out:
            add("  (no output recorded)")
            continue

        for k, v in out.items():
            v_clean = _norm(v)

            if isinstance(v_clean, dict):
                add(f"  {k}:")
                for kk, vv in v_clean.items():
                    add(f"    {kk}: {vv}")
            else:
                add(f"  {k}: {v_clean}")

    # Final shape
    shape = summary.get('cleaned dataframe shape', None)
    if shape:
        add("\nFinal dataframe shape:")
        add(f"  rows: {_norm_scalar(shape[0])}")
        add(f"  cols: {_norm_scalar(shape[1])}")

    return "\n".join(lines)

# Test set 1 - testing validation of the clean_categories map (complex)

df_config_map_test = pd.DataFrame({
 'status': ['open', 'closed', 'pending'],
 'gender': ['m', 'f', 'unknown'],
 'region': ['uk', 'england', 'scotland']})

config_map_valid = {
 'categories': {'col_list': ['status', 'gender', 'region'],
                'strategy': 'lower'},
                'map': {'status': {'open': 'Open', 'closed': 'Closed', 'pending': 'Pending'},
                        'gender': {'m': 'Male', 'f': 'Female', 'unknown': 'Unknown'},
                        'region': {'uk': 'United Kingdom', 'england': 'England', 'scotland': 'Scotland'}}}

config_map_not_dict = {
 'categories': {'col_list': ['status'],
                'strategy': 'lower',
                'map': 123}}

config_map_bad_outer_key = {
 'categories': {'col_list': ['status'],
                'strategy': 'lower',
                'map': {'ghost_col': {'open': 'Open'}}}}

config_map_outer_value_not_dict = {
 'categories': {'col_list': ['status'],
                'strategy': 'lower',
                'map': {
                'status': 'Open'}}}

config_map_empty_inner = {
 'categories': {'col_list': ['status'],
                'strategy': 'lower',
                'map': {'status': {}}}}

config_map_mismatched_types = {
 'categories': {'col_list': ['status'],
                'strategy': 'lower',
                'map': {'status': {'open': 1, 'closed': 'Closed'}}}}

config_map_non_string_key = {
 'categories': {'col_list': ['status'],
                'strategy': 'lower',
                'map': {'status': {1: 'Open', 'closed': 'Closed'}}}}

config_map_non_string_value = {
 'categories': {'col_list': ['status'],
                'strategy': 'lower',
                'map': {'status': {'open': 1, 'closed': 'Closed'}}}}

config_map_multiple_errors = {
 'categories': {'col_list': ['status', 'gender'],
                'strategy': 'lower',
                'map': {'status': {'open': 1, 2: 'Closed'}, 'gender': {}}}}

config_map_extra_columns = {
 'categories': {'col_list': ['status', 'gender'],
                'strategy': 'lower',
                'map': {'status': {'open': 'Open'}, 'gender': {'m': 'Male'}, 'region': {'uk': 'United Kingdom'}}}}

config_map_inner_not_dict = {
 'categories': {'col_list': ['status', 'gender'],
                'strategy': 'lower',
                'map': {'status': {'open': 'Open'}, 'gender': 'Male'}}}

# Test set 2 - testing validation of the full config dictionary

# Test set 2 part A: Testing for bad config function tags, tags with no parameter dictionaries and parameter dictionaries with invalid parameters

df_config_test_small = pd.DataFrame({
 'num_col': [1, 2, 3],
 'str_col': ['a', 'b', 'c']})

config_bad_tags = {
 'string': {'strategy': 'strip'},
 'dates': {'strategy': 'median'},
 'not_a_tag': {'foo': 'bar'},      # invalid tag
 'missing': {'col_dict': {'num_col': 'number'}}}

config_bad_param_types = {
 'string': 'strip',                # should be dict
 'dates': 123,                     # should be dict
 'missing': {'col_dict': {'num_col': 'number'}},
 'numeric': None}                  # should be dict

config_bad_param_names = {
 'string': {'strategy': 'strip', 'wrong_param': True},
 'dates': {'strategy': 'median', 'invalid': 123},
 'missing': {'col_dict': {'num_col': 'number'}, 'badname': 'oops'},
 'numeric': {'col_list': ['num_col'], 'empty': 'not_valid'}}

# Test set 2 part B: Testing for bad individual parameters (complex)

df_config_test_stress = pd.DataFrame({
 'num_col': [1, 2, 1000, -999],
 'str_col': ['Hello', 'N/A', '???', 'world'],
 'date_col': [pd.Timestamp('2020-01-01'),
              pd.Timestamp('1970-01-01'),
              pd.Timestamp('2026-09-16'),
              pd.NaT],
 'bool_col': [True, False, True, None]})

config_stress_clean = {
 'duplicates': {'col_list': ['ghost_col']}, # invalid value: column doesn't exist
 'string': {'strategy': 'explode'},         # invalid strategy
 'dates': {'strategy': 123},                # invalid type
 'missing': {
    'col_dict': {'ghost_col': 'number'},   # invalid column
    'strat_dict': {'number': 'warpdrive'}, # invalid strategy
    'val_dict': {'number': ['oops']}},     # invalid token type
 'numeric': {
    'col_list': ['str_col'],               # invalid: not numeric
    'strategy': 'void',                    # invalid strategy
    'outlier': 999},                       # invalid outlier value
 'outliers': {
    'col_list': ['date_col'],              # invalid: not numeric
    'method': 'unknown'},                  # invalid method
 'categories': {
    'col_list': ['num_col'],               # invalid: not category/string
    'strategy': 'reverse'},                # invalid strategy
 'scaling': {
    'col_list': ['str_col'],               # invalid: not numeric
    'strategy': 'log'}}                    # invalid strategy

# Test set 3 - general testing of the processing section of clean_pipeline (tests are representative only for demo purposes)

df_processing_test = pd.DataFrame({
 'name':        ['Alice', 'BOB', 'NaN', 'charlie', 'DAVE', 'Alice'],   # duplicate
 'signup_date': ['2024-01-01', '01/02/2024', None, 'March 3 2024', 'not a date', '2024-01-01'],
 'score':       [10.0, None, 30.0, None, 50.0, 10.0],
 'mixed_num':   ['10', '20.5', None, 'invalid', '40', '10'],
 'outlier_col': [10, 12, 11, 9, 500, 10],
 'category':    ['Red', 'RED', 'blue', None, 'Blue', 'Red'],
 'scale_me':    [1, 2, 3, 4, 5, 1],
 'notes':       ['ok', 'fine', 'check', None, 'done', 'ok']})

config_processing_test = {
    'string':     {'strategy': 'lower'},
    'dates':      {'strategy': 'median'},
    'missing':    {'strat_dict': {'number': 'median', 'string': '', 'date': 'today', 'bool': False}},
    'numeric':    {'strategy': 'mean', 'outlier': 3},
    'outliers':   {'method': 'iqr', 'col_list': ['outlier_col']},
    'categories': {'strategy': 'lower', 'map': {'category': {'red': 'Red', 'blue': 'Blue'}}},
    'scaling':    {'strategy': 'minmax'},
    'duplicates': {'col_list': ['name']}}

# Test validation of clean_categories map

#print(df_config_map_test)
#test_config = config_map_valid
#test_config = config_map_not_dict
#test_config = config_map_bad_outer_key
#test_config = config_map_outer_value_not_dict
#test_config = config_map_empty_inner
#test_config = config_map_mismatched_types
#test_config = config_map_non_string_key
#test_config = config_map_non_string_value
#test_config = config_map_multiple_errors
#test_config = config_map_extra_columns
#test_config = config_map_inner_not_dict
#print(test_config)
#clean_pipeline(df=df_config_map_test, config=test_config);

# Test validation of bad function tags, tags with no parameter dictionaries and bad parameter names

#print(df_config_test_small)
#test_config = config_bad_tags
#test_config = config_bad_param_types
#test_config = config_bad_param_names
#print(test_config)
#clean_pipeline(df=df_config_test_small, config=test_config);

# Stress test of individual parameters

#print(df_config_test_stress)
#print(config_stress_clean)
#clean_pipeline(df=df_config_test_stress, config=config_stress_clean);

# Test pipeline processing

print(df_processing_test)
print(config_processing_test)
df_clean, summary = clean_pipeline(df=df_processing_test, config=config_processing_test)
print(df_clean)
print(format_pipeline_summary(summary))